### Es 1
Hai a disposizione un file `data.csv` contenente dati mensili di traffico aereo con due colonne:

- `date`: data in formato `YYYY-MM` (mese/anno)
- `passengers`: numero di passeggeri per quel mese


Costruisci un modello di **regressione polinomiale** che approssima l’andamento del numero di passeggeri nel tempo.

1. Carica il dataset.
2. Convertilo in un formato numerico utilizzando una colonna `mese_numerico` che conti i mesi a partire da gennaio 1949.
3. Applica una regressione polinomiale (grado a tua scelta).
4. Calcola l’RMSE tra i valori reali e quelli predetti.
5. Visualizza i dati reali e la curva stimata con Plotly.

In [19]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
import plotly.graph_objects as go
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

In [20]:
# Step 1: caricamento dataset

df = pd.read_csv("/Users/sofia/Desktop/python/LabProg_2/data.csv")
df = df.dropna()        # rimuove le righe che non contengono valori (NaN)
df

,date,passengers
0,1949-01,112.0
1,1949-02,118.0
2,1949-03,132.0
3,1949-04,129.0
4,1949-05,121.0
...,...,...
125,1960-07,622.0
126,1960-08,606.0
127,1960-09,508.0
128,1960-10,461.0


In [21]:
# Step 2: convertire dataset in formato numerico usando una colonna `mese_numerico` che conti i mesi da gennaio 1949
df['data'] = pd.to_datetime(df['date'], format='%Y-%m')   # con formato gli dico che voglio anno - mese
df['mese_numerico'] = (df['data'].dt.year - 1949) * 12 + (df['data'].dt.month - 1)
df

# moltiplicazione per 12 perche estrai il numero del mese in tutti gli anni considerati
# per estrarre: colonna.dt.cosa_vuoi_estrarre

,date,passengers,data,mese_numerico
0,1949-01,112.0,1949-01-01,0
1,1949-02,118.0,1949-02-01,1
2,1949-03,132.0,1949-03-01,2
3,1949-04,129.0,1949-04-01,3
4,1949-05,121.0,1949-05-01,4
...,...,...,...,...
125,1960-07,622.0,1960-07-01,138
126,1960-08,606.0,1960-08-01,139
127,1960-09,508.0,1960-09-01,140
128,1960-10,461.0,1960-10-01,141


In [22]:
# Step 3: applicare una regressione polinomiale (grado a scelta), ho scelto di grado 2
X = df['mese_numerico'].values
Y = df['passengers'].values
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, shuffle=False)

# test_size=0.2 siginfica che il 20% dei dati va nel test
# shuffle=False lascia in ordine i dati, visto che ho un dataset temporale

reg_grado2 = make_pipeline(PolynomialFeatures(2), LinearRegression())
reg_grado2.fit(X_train.reshape(-1,1), Y_train)
Y_pred = reg_grado2.predict(X_test.reshape(-1,1))

In [23]:
# Step 4: calcolare l’rmse
rmse = root_mean_squared_error(Y_test, Y_pred)

print('rmse: ', rmse)

rmse:  72.2196458281994


In [24]:
# Step 5: visualizzare i dati reali e la curva stimata con Plotly
fig = go.Figure()

# Dati reali: sono dei punti
fig.add_trace(go.Scatter(
    x=X_test,
    y=Y_test,
    mode='markers',
    name='Dati reali',
    line=dict(color='blue')
))

# Curva stimata: è una linea
fig.add_trace(go.Scatter(
    x=X_test,
    y=Y_pred,
    mode='lines',
    name='Curva stimata',
    line=dict(color='red')
))

fig.update_layout(
    title='Regressione polinomiale - Passeggeri',
    xaxis_title='Mese numerico',
    yaxis_title='Passeggeri'
)

fig.show()